# Notebook 3 — Agent Demo

This notebook loads the LangChain agent defined in `agent/agent.py` and runs a set of example queries that demonstrate the full pipeline in action.

Each query shows:
- Which tools the agent decided to invoke
- What data those tools returned
- The final synthesised answer

This is the showpiece notebook — the outputs here are what goes into your report appendix.

> Prerequisite: `02_pipeline.ipynb` must have run. Docker services must be up.

## 0. Setup — load the agent

In [9]:
import sys, os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')

# Add the project root to the Python path so we can import agent/
sys.path.insert(0, os.path.abspath('..'))

from agent.agent import build_agent

# Build the agent — this wires together the LLM, tools, and system prompt
agent_executor = build_agent(verbose=True)

print('Agent loaded and ready')

Agent loaded and ready


---
## 1. Helper — run a query and display the reasoning trace

We wrap `agent_executor.invoke()` in a small helper that prints the intermediate steps clearly. This makes the agent's reasoning transparent for the report.

In [10]:
def run_query(question: str):
    """Run a question through the agent and print the full reasoning trace."""
    print('=' * 60)
    print(f'QUERY: {question}')
    print('=' * 60)

    result = agent_executor.invoke({'input': question})

    # Intermediate steps show which tools were called and what they returned
    print('\nREASONING TRACE:')
    for step in result.get('intermediate_steps', []):
        action, observation = step
        print(f'  Tool called : {action.tool}')
        print(f'  Input       : {action.tool_input}')
        print(f'  Output      : {str(observation)[:300]}...')
        print()

    print('FINAL ANSWER:')
    print(result['output'])
    print()
    return result

---
## 2. Demo Query 1 — Price trend analysis

A purely structured query. The agent should call `query_price_data` only, using SQL to compute the answer from PostgreSQL.

In [11]:
r1 = run_query(
    'What is the 30-day average closing price and current RSI for NVIDIA (NVDA)? '
    'Is it currently overbought or oversold?'
)

QUERY: What is the 30-day average closing price and current RSI for NVIDIA (NVDA)? Is it currently overbought or oversold?

> Calling tool : query_price_data
  Input        : {'sql': "SELECT AVG(close) as avg_close, rsi_14 FROM prices WHERE ticker='NVDA' AND date >= NOW() - INTERVAL '30 days'"}
  Output       : SQL error: (psycopg2.errors.GroupingError) column "prices.rsi_14" must appear in the GROUP BY clause or be used in an aggregate function
LINE 1: SELECT AVG(close) as avg_close, rsi_14 FROM prices WHERE tic...
                                        ^

[SQL: SELECT AVG(close) as avg_close, rsi_14 FRO

> Calling tool : query_price_data
  Input        : {'sql': "SELECT AVG(close) as avg_close, rsi_14 FROM prices WHERE ticker='NVDA' AND date >= NOW() - INTERVAL '30 days' GROUP BY rsi_14 ORDER BY date DESC LIMIT 1"}
  Output       : SQL error: (psycopg2.errors.GroupingError) column "prices.date" must appear in the GROUP BY clause or be used in an aggregate function
LINE 1: ...W() - I

---
## 3. Demo Query 2 — Sentiment vs price divergence

A multi-tool query requiring both structured price data and unstructured news sentiment. The agent should call `query_price_data` AND `search_financial_news` and synthesise the results.

In [12]:
r2 = run_query(
    'Is there a divergence between NVIDIA news sentiment and its 30-day price momentum? '
    'Summarise what the news is saying and compare it to the price trend.'
)

QUERY: Is there a divergence between NVIDIA news sentiment and its 30-day price momentum? Summarise what the news is saying and compare it to the price trend.

> Calling tool : search_financial_news
  Input        : {'query': 'NVIDIA news sentiment', 'ticker': 'NVDA'}
  Output       : {
  "results": []
}

> Calling tool : query_price_data
  Input        : {'sql': "SELECT price_change_pct FROM prices WHERE ticker = 'NVDA' AND date >= NOW() - INTERVAL '30 days'"}
  Output       : {
  "rows": [
    {
      "price_change_pct": "1.4052"
    },
    {
      "price_change_pct": "-5.4561"
    },
    {
      "price_change_pct": "-4.1646"
    },
    {
      "price_change_pct": "2.9855"
    },
    {
      "price_change_pct": "-1.3317"
    },
    {
      "price_change_pct": "1.6606"
  

> Final answer after 3 iterations

REASONING TRACE:
  Tool called : search_financial_news
  Input       : {'query': 'NVIDIA news sentiment', 'ticker': 'NVDA'}
  Output      : {
  "results": []
}...

  Tool called : 

---
## 4. Demo Query 3 — SEC filing risk analysis

A document-retrieval query. The agent should call `search_financial_news` (which searches SEC filings too) and `get_filing_metadata` to ground its answer in disclosed risk factors.

In [13]:
r3 = run_query(
    'What key risks did Apple (AAPL) disclose in its most recent 10-K filing? '
    'Cross-reference these risks with AAPL price volatility over the last 3 months.'
)

QUERY: What key risks did Apple (AAPL) disclose in its most recent 10-K filing? Cross-reference these risks with AAPL price volatility over the last 3 months.

> Calling tool : search_financial_news
  Input        : {'query': 'Apple 10-K risk factors', 'ticker': 'AAPL'}
  Output       : {
  "results": []
}

> Calling tool : query_price_data
  Input        : {'sql': "SELECT date, close, price_change_pct FROM prices WHERE ticker = 'AAPL' AND date >= NOW() - INTERVAL '3 months' ORDER BY date ASC"}
  Output       : {
  "rows": [
    {
      "date": "2025-12-29",
      "close": "273.7600",
      "price_change_pct": "0.1317"
    },
    {
      "date": "2025-12-30",
      "close": "273.0800",
      "price_change_pct": "-0.2484"
    },
    {
      "date": "2025-12-31",
      "close": "271.8600",
      "price_chang

> Calling tool : get_filing_metadata
  Input        : {'form_type': '10-K', 'ticker': 'AAPL'}
  Output       : {
  "ticker": "AAPL",
  "company_name": "Apple Inc.",
  "cik": "000032

---
## 5. Demo Query 4 — Sector comparison

A broader analytical query requiring the agent to aggregate across multiple tickers. Tests whether the agent constructs correct SQL GROUP BY logic.

In [14]:
r4 = run_query(
    'Compare the 90-day price performance of the banking sector (JPM, GS, BAC) '
    'versus the tech sector (AAPL, MSFT, NVDA). Which sector has shown stronger momentum?'
)

QUERY: Compare the 90-day price performance of the banking sector (JPM, GS, BAC) versus the tech sector (AAPL, MSFT, NVDA). Which sector has shown stronger momentum?

> Calling tool : query_price_data
  Input        : {'sql': "SELECT ticker, AVG(price_change_pct) as avg_price_change FROM prices WHERE date >= NOW() - INTERVAL '90 days' AND (ticker = 'JPM' OR ticker = 'GS' OR ticker = 'BAC') GROUP BY ticker"}
  Output       : {
  "rows": [
    {
      "ticker": "JPM",
      "avg_price_change": "-0.18228474576271186441"
    },
    {
      "ticker": "BAC",
      "avg_price_change": "-0.24835762711864406780"
    },
    {
      "ticker": "GS",
      "avg_price_change": "-0.11540169491525423729"
    }
  ],
  "count": 3
}

> Calling tool : query_price_data
  Input        : {'sql': "SELECT ticker, AVG(price_change_pct) as avg_price_change FROM prices WHERE date >= NOW() - INTERVAL '90 days' AND (ticker = 'AAPL' OR ticker = 'MSFT' OR ticker = 'NVDA') GROUP BY ticker"}
  Output       : {
  "rows"

---
## 6. Hallucination check — verifying agent citations

A key concern with LLM agents is hallucination — generating plausible-sounding but ungrounded answers. We verify that the agent's factual claims can be traced back to a specific tool call and data source.

In [15]:
# Extract all tool calls from the four demo queries and verify each has a traceable source
all_results = [r1, r2, r3, r4]
query_labels = ['Price trend', 'Sentiment divergence', 'SEC risk', 'Sector comparison']

print('TOOL USAGE AUDIT')
print('=' * 60)
for label, result in zip(query_labels, all_results):
    steps = result.get('intermediate_steps', [])
    tools_used = [s[0].tool for s in steps]
    print(f'{label}')
    print(f'  Tools invoked : {tools_used}')
    print(f'  Steps         : {len(steps)}')
    # A response with zero tool calls is a red flag — the agent answered from training data,
    # not from the pipeline, which counts as hallucination in this context
    if not tools_used:
        print('  WARNING: no tools called — answer may not be grounded in pipeline data')
    print()

TOOL USAGE AUDIT
Price trend
  Tools invoked : ['query_price_data', 'query_price_data', 'query_price_data', 'query_price_data']
  Steps         : 4

Sentiment divergence
  Tools invoked : ['search_financial_news', 'query_price_data']
  Steps         : 2

SEC risk
  Tools invoked : ['search_financial_news', 'query_price_data', 'get_filing_metadata']
  Steps         : 3

Sector comparison
  Tools invoked : ['query_price_data', 'query_price_data']
  Steps         : 2



---
## 7. Interactive query cell

Use this cell to test your own questions against the agent.

In [16]:
# Change this question and re-run the cell to test any query
your_question = 'Which stock in our watchlist has the highest RSI right now?'

run_query(your_question)

QUERY: Which stock in our watchlist has the highest RSI right now?

> Calling tool : query_price_data
  Input        : {'sql': 'SELECT ticker, rsi_14 FROM prices WHERE date = (SELECT MAX(date) FROM prices) ORDER BY rsi_14 DESC LIMIT 1'}
  Output       : {
  "rows": [
    {
      "ticker": "NVDA",
      "rsi_14": "46.3677"
    }
  ],
  "count": 1
}

> Calling tool : query_price_data
  Input        : {'sql': 'SELECT ticker, MAX(rsi_14) as max_rsi FROM prices WHERE date = (SELECT MAX(date) FROM prices) GROUP BY ticker ORDER BY max_rsi DESC LIMIT 1'}
  Output       : {
  "rows": [
    {
      "ticker": "NVDA",
      "max_rsi": "46.3677"
    }
  ],
  "count": 1
}

> Final answer after 3 iterations

REASONING TRACE:
  Tool called : query_price_data
  Input       : {'sql': 'SELECT ticker, rsi_14 FROM prices WHERE date = (SELECT MAX(date) FROM prices) ORDER BY rsi_14 DESC LIMIT 1'}
  Output      : {
  "rows": [
    {
      "ticker": "NVDA",
      "rsi_14": "46.3677"
    }
  ],
  "count": 1
}..

{'input': 'Which stock in our watchlist has the highest RSI right now?',
 'output': 'The stock with the highest RSI right now is NVDA, with an RSI of 46.3677.',
 'intermediate_steps': [(<agent.agent.ToolAction at 0x2656d579050>,
   '{\n  "rows": [\n    {\n      "ticker": "NVDA",\n      "rsi_14": "46.3677"\n    }\n  ],\n  "count": 1\n}'),
  (<agent.agent.ToolAction at 0x2656d560590>,
   '{\n  "rows": [\n    {\n      "ticker": "NVDA",\n      "max_rsi": "46.3677"\n    }\n  ],\n  "count": 1\n}')]}